# Fase 1: limpieza de datos

Este notebook sirve para **explorar y probar** la limpieza con los datos simulados
(`data/simulated/`). Cuando una etapa funcione, su código se pasa a
`src/occupancy/cleaning.py`.

Orden de las etapas (ver el documento del pipeline):

0. Cargar los datos y convertir las fechas
1. Duplicados
2. Control de rango
3. Picos aislados
4. Huecos
5. Suavizado (media móvil)

Las celdas marcadas como **Comprobación** ya están escritas: ejecútalas después de
tu código para saber si el resultado es correcto.

In [1]:
import json
from pathlib import Path

import pandas as pd

# Buscar la raíz del repositorio (la carpeta que contiene config.yaml)
ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "simulated" / "readings.json"
FAULTS_PATH = ROOT / "data" / "simulated" / "faults.json"
print("Datos:", DATA_PATH)

Datos: c:\Users\danib\Desktop\ERASMUS\IOT\loudness-classifier\data\simulated\readings.json


## 0. Cargar los datos

El archivo tiene el mismo formato que la respuesta de la API. Esta celda ya lo lee y
deja en `data` un diccionario con una lista de lecturas por sensor.

In [2]:
with open(DATA_PATH, encoding="utf-8") as f:
    response = json.load(f)

data = response["data"]
print("Sensores:", list(data))
print("Primera lectura de temp:", data["temp"][0])
print("Primera lectura de PIR:", data["PIR"][0])

Sensores: ['PIR', 'pressure', 'hum', 'temp', 'co2equivalent', 'voc', 'airresistance', 'mic_vol', 'busy_level']
Primera lectura de temp: {'timestamp': '2026-10-20 09:00:00.000', 'value': 21.01}
Primera lectura de PIR: {'timestamp': '2026-10-20 09:00:00.000', 'values': [{'ID': 'pir-1', 'value': 1}, {'ID': 'pir-2', 'value': 1}, {'ID': 'pir-3', 'value': 1}]}


### Tu tarea: convertir cada sensor en una tabla

Guarda el resultado en un diccionario llamado **`tables`**: una tabla por sensor
(menos `PIR` y `busy_level`), con las columnas `timestamp` y `value`.

- Cada lista de lecturas se convierte en tabla con `pd.DataFrame`.
- Convierte `timestamp` a fecha con `pd.to_datetime`, indicando el formato
  `"%Y-%m-%d %H:%M:%S.%f"`.

In [3]:
# Tu código aquí
tables = {}
for sensor, readings in data.items():
    if sensor in ("PIR", "busy_level"):
        continue
    table = pd.DataFrame(readings)
    table["timestamp"] = pd.to_datetime(table["timestamp"], format="%Y-%m-%d %H:%M:%S.%f")
    tables[sensor] = table

for i, j in tables.items():
    print(f"Tabla de {i}:")
    print(j.head())


Tabla de pressure:
            timestamp    value
0 2026-10-20 09:00:00  1012.70
1 2026-10-20 09:00:10  1012.93
2 2026-10-20 09:00:20  1013.26
3 2026-10-20 09:00:30  1013.01
4 2026-10-20 09:00:40  1012.67
Tabla de hum:
            timestamp  value
0 2026-10-20 09:00:00  44.99
1 2026-10-20 09:00:10  45.25
2 2026-10-20 09:00:20  45.22
3 2026-10-20 09:00:30  45.32
4 2026-10-20 09:00:40  45.29
Tabla de temp:
            timestamp  value
0 2026-10-20 09:00:00  21.01
1 2026-10-20 09:00:10  21.05
2 2026-10-20 09:00:20  21.05
3 2026-10-20 09:00:30  21.06
4 2026-10-20 09:00:40  21.12
Tabla de co2equivalent:
            timestamp  value
0 2026-10-20 09:00:00  462.9
1 2026-10-20 09:00:10  471.7
2 2026-10-20 09:00:20  467.8
3 2026-10-20 09:00:30  472.0
4 2026-10-20 09:00:40  479.9
Tabla de voc:
            timestamp  value
0 2026-10-20 09:00:00  0.672
1 2026-10-20 09:00:10  0.679
2 2026-10-20 09:00:20  0.693
3 2026-10-20 09:00:30  0.683
4 2026-10-20 09:00:40  0.698
Tabla de airresistance:
        

### Tu tarea: convertir los PIR en una tabla

Guarda el resultado en una tabla llamada **`pir`**, con una fila por marca de tiempo y
las columnas `timestamp`, `pir-1`, `pir-2` y `pir-3`. Convierte también `timestamp`
a fecha.

In [4]:
# Tu código aquí
rows = []
for reading in data["PIR"]:
    row = {"timestamp": pd.to_datetime(reading["timestamp"], format="%Y-%m-%d %H:%M:%S.%f")}
    for p in reading["values"]:
        # añade a row la clave p["ID"] con el valor p["value"]
        row[p["ID"]] = p["value"]
    # añade row a la lista rows
    print(row)
    rows.append(row)

pir = pd.DataFrame(rows)
print(pir.head())
# convierte la columna timestamp a fecha

{'timestamp': Timestamp('2026-10-20 09:00:00'), 'pir-1': 1, 'pir-2': 1, 'pir-3': 1}
{'timestamp': Timestamp('2026-10-20 09:00:10'), 'pir-1': 1, 'pir-2': 1, 'pir-3': 1}
{'timestamp': Timestamp('2026-10-20 09:00:20'), 'pir-1': 1, 'pir-2': 1, 'pir-3': 1}
{'timestamp': Timestamp('2026-10-20 09:00:30'), 'pir-1': 1, 'pir-2': 1, 'pir-3': 1}
{'timestamp': Timestamp('2026-10-20 09:00:40'), 'pir-1': 0, 'pir-2': 1, 'pir-3': 1}
{'timestamp': Timestamp('2026-10-20 09:00:50'), 'pir-1': 1, 'pir-2': 1, 'pir-3': 1}
{'timestamp': Timestamp('2026-10-20 09:01:00'), 'pir-1': 1, 'pir-2': 1, 'pir-3': 1}
{'timestamp': Timestamp('2026-10-20 09:01:10'), 'pir-1': 1, 'pir-2': 1, 'pir-3': 1}
{'timestamp': Timestamp('2026-10-20 09:01:20'), 'pir-1': 1, 'pir-2': 1, 'pir-3': 1}
{'timestamp': Timestamp('2026-10-20 09:01:30'), 'pir-1': 1, 'pir-2': 1, 'pir-3': 1}
{'timestamp': Timestamp('2026-10-20 09:01:40'), 'pir-1': 1, 'pir-2': 1, 'pir-3': 1}
{'timestamp': Timestamp('2026-10-20 09:01:50'), 'pir-1': 1, 'pir-2': 1, 'pir

In [5]:
# Comprobación de la carga (no hace falta modificarla)
expected = {s: 1067 for s in ["pressure", "hum", "temp", "voc", "airresistance", "mic_vol"]}
expected["co2equivalent"] = 1046

for sensor, n in expected.items():
    table = tables[sensor]
    ok = len(table) == n and pd.api.types.is_datetime64_any_dtype(table["timestamp"])
    print(f"{sensor:14} filas={len(table):5}  esperado={n:5}  {'OK' if ok else 'REVISAR'}")

ok_pir = (len(pir) == 1067
          and set(pir.columns) == {"timestamp", "pir-1", "pir-2", "pir-3"}
          and pd.api.types.is_datetime64_any_dtype(pir["timestamp"]))
print(f"{'PIR':14} filas={len(pir):5}  esperado= 1067  {'OK' if ok_pir else 'REVISAR'}")

pressure       filas= 1067  esperado= 1067  OK
hum            filas= 1067  esperado= 1067  OK
temp           filas= 1067  esperado= 1067  OK
voc            filas= 1067  esperado= 1067  OK
airresistance  filas= 1067  esperado= 1067  OK
mic_vol        filas= 1067  esperado= 1067  OK
co2equivalent  filas= 1046  esperado= 1046  OK
PIR            filas= 1067  esperado= 1067  OK


## 1.1 Duplicados

Un duplicado de QoS 1 es una fila **exactamente igual** a otra: misma hora y mismos
valores. Quítalos con `drop_duplicates` sobre la fila completa, no solo sobre la hora.

Guarda el resultado en las mismas variables (`tables` y `pir`).

In [6]:
# Tu código aquí
for sensor, table in tables.items():
    tables[sensor] = table.drop_duplicates()

pir= pir.drop_duplicates()

In [7]:
# Comprobación de duplicados (no hace falta modificarla)
expected = {s: 1059 for s in ["pressure", "hum", "temp", "voc", "airresistance", "mic_vol"]}
expected["co2equivalent"] = 1038

for sensor, n in expected.items():
    table = tables[sensor]
    ok = len(table) == n and table.duplicated().sum() == 0
    print(f"{sensor:14} filas={len(table):5}  esperado={n:5}  {'OK' if ok else 'REVISAR'}")

ok_pir = len(pir) == 1059 and pir.duplicated().sum() == 0
print(f"{'PIR':14} filas={len(pir):5}  esperado= 1059  {'OK' if ok_pir else 'REVISAR'}")

pressure       filas= 1059  esperado= 1059  OK
hum            filas= 1059  esperado= 1059  OK
temp           filas= 1059  esperado= 1059  OK
voc            filas= 1059  esperado= 1059  OK
airresistance  filas= 1059  esperado= 1059  OK
mic_vol        filas= 1059  esperado= 1059  OK
co2equivalent  filas= 1038  esperado= 1038  OK
PIR            filas= 1059  esperado= 1059  OK


In [8]:
for sensor, table in tables.items():
    print(sensor, type(table).__name__)

pressure DataFrame
hum DataFrame
temp DataFrame
co2equivalent DataFrame
voc DataFrame
airresistance DataFrame
mic_vol DataFrame


## 1.2 Control de rango

Convierte en vacío (`NaN`) los valores imposibles para cada sensor, **sin borrar la
fila**. Los límites reales saldrán de la hoja de datos de cada sensor; para los datos
simulados basta con detectar los valores claramente imposibles.

Herramienta: `mask`.

In [9]:
# Tu código aquí
Limits={
    "temp": {"min": -10, "max": 50},
    "hum": {"min": 0, "max": 100},
    "voc": {"min": 0, "max": float('inf')},
    "co2equivalent": {"min": 0, "max": 5000},
    "airresistance": {"min": 0, "max": float('inf')},
    "pressure": {"min": 950, "max": 1100},
    "mic_vol": {"min": 0, "max": 1},
}

for sensor, table in tables.items():
    limits = Limits[sensor]
    condition = (table["value"] < limits["min"]) | (table["value"] > limits["max"])
    tables[sensor]["value"]  = table["value"].mask(condition)

    



## 1.3 Picos aislados

Convierte en vacío los valores que se disparan un instante y vuelven, comparando cada
valor con sus vecinos. Sobre todo en `mic_vol`.

Herramientas: `rolling` y `mask`.

In [10]:
# Tu código aquí

## 1.4 Huecos

Primero mide cada cuánto llegan las lecturas (`diff` sobre `timestamp`). Después
rellena solo los huecos cortos y deja vacíos los largos.

Herramienta: `interpolate` con `method="time"` y un límite.

In [11]:
# Tu código aquí

## 1.5 Suavizado

Media móvil que **solo mira hacia atrás**, aplicada a `mic_vol`, `co2equivalent` y `voc`.

Herramienta: `rolling` con una ventana de tiempo.

In [12]:
# Tu código aquí

## Inspección visual

Dibuja cada sensor antes y después de la limpieza para comprobar que no queda nada raro.
Es la parte de "inspección humana" de la diapositiva 25.

In [13]:
# Tu código aquí